In [ ]:
import pandas as pd    
import numpy as np                  
import seaborn as sns  
import matplotlib.pyplot as plt            


In [ ]:
df = pd.read_csv("mydrone_Data.csv")

In [ ]:
df.columns

In [ ]:
df.columns[df.columns.str.startswith("volt")]

In [ ]:
df.info()

In [ ]:
df.head(3)

In [ ]:
excluded_cols = ['voltageCell1', 
                 'voltageCell2', 
                 'voltageCell3',
                    'voltageCell4', 
                    'voltageCell5', 
                    'voltageCell6',
                    'message',
                    'datetime(utc)'
                    ]

In [ ]:
df_clean = df.drop(columns=excluded_cols)

In [ ]:
df_clean.head(2)

In [ ]:
# df_clean["isphoto"]

In [ ]:
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(" ", "_")

In [ ]:
selected_columns = ['time(millisecond)',
                    'latitude', 'longitude',
       'height_above_takeoff(feet)',
       'height_above_ground_at_drone_location(feet)',
       'ground_elevation_at_drone_location(feet)',
       'altitude_above_sealevel(feet)', 
       'height_sonar(feet)', 
       'speed(mph)',
       'distance(feet)', 
       'mileage(feet)', 'satellites', 'gpslevel',
       'voltage(v)', 'max_altitude(feet)', 'max_ascent(feet)',
       'max_speed(mph)', 'max_distance(feet)', 'xspeed(mph)', 'yspeed(mph)',
       'zspeed(mph)', 'compass_heading(degrees)', 'pitch(degrees)',
       'roll(degrees)', 'isphoto', 'isvideo', 'rc_elevator', 'rc_aileron',
       'rc_throttle', 'rc_rudder', 'rc_elevator(percent)',
       'rc_aileron(percent)', 'rc_throttle(percent)', 'rc_rudder(percent)',
       'gimbal_heading(degrees)', 'gimbal_pitch(degrees)',
       'gimbal_roll(degrees)', 'battery_percent', 'current(a)',
       'battery_temperature(f)', 'altitude(feet)', 'ascent(feet)',
       'flycstateraw'
       ]

df_clean.loc[:,selected_columns]

In [ ]:
df_clean[selected_columns]

In [ ]:
df_clean.head(3)

In [ ]:
X = df_clean.drop(columns=["flycstate"])
y = df_clean["flycstate"]

In [ ]:
X.shape

In [ ]:
print(len(X.columns))

In [ ]:
X.columns


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import joblib


In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [ ]:
print(np.unique(y))

In [ ]:
encoder.classes_

In [ ]:
dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y,                                                   
                                                    test_size=0.2,
                                                    random_state=42)

model = XGBClassifier()
pipe = Pipeline([("preprocess", StandardScaler()),
                 ("xgboost", model)])

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
# # Save the Model
# joblib.dump(pipe, "classifier_model.pkl")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = pipe.predict(X_test)
report = classification_report(y_test, y_pred)

In [ ]:
print(report)

In [ ]:
conf_mat = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_mat, fmt=".2f", annot=True)
plt.show()

In [ ]:
# load the model
new_model = joblib.load("classifier_model.pkl")
y_prediction = new_model.predict(X_test)

In [ ]:
y_prediction